# 🔧 Function Calling Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/llm-api-hands-on/blob/main/1_Text_Generation/04-Function%20Calling-Practice.ipynb)

> 정답 코드는 [`Practice_solution/04-Function Calling-Practice_solution.ipynb`](../Practice_solution/04-Function%20Calling-Practice_solution.ipynb)에 있습니다.

> **주의:** Colab 런타임을 재시작하면 환경 변수가 초기화되므로 `OPENAI_API_KEY`를 다시 설정해야 합니다.

## 1) 환경 준비 및 API 키 설정

In [ ]:
%pip install -q -U openai requests

### API 키 설정 및 Client 생성
- **Colab**: 왼쪽 메뉴 `보안 비밀(열쇠 아이콘)`에 `OPENAI_API_KEY`를 저장하고 **노트북 액세스**를 켜 두면 자동으로 불러옵니다.
- **로컬**: 환경 변수 `OPENAI_API_KEY`를 미리 설정해 두거나, 아래 셀을 실행했을 때 뜨는 입력창에 키를 붙여넣으면 됩니다.
- 아래의 코드에서 오류가 난다면 API 키의 오류일 가능성이 높습니다.

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

In [ ]:
from openai import OpenAI

client = OpenAI()
print("✅ 클라이언트 생성 완료")

## Tool Choice 제어 및 Function Calling 종합 실습: 날씨 → 활동 추천 자동 체인

**실습 목표**
- OpenAI Function Calling의 실제 사용 흐름을 이해한다.
- 모델이 **여러 개의 툴을 순차적으로 호출하는 과정(Chained Tool Calling)** 을 학습한다.
- 기온 조회 → 활동 추천 → 최종 자연어 응답까지 이어지는 전체 파이프라인을 구현한다.

**실습 구성**

1. **get_weather**
   - 입력: `city`
   - 출력: 현재 기온
   - 실제 API(Open-Meteo)로 기온을 조회
   - Step 1에서는 이 툴만 사용 가능하도록 `tool_choice` 제한

2. **recommend_activity**
   - 입력: `temperature`
   - 출력: 기온 구간별 추천 활동
   - Step 2에서만 호출되도록 `tool_choice`로 제어

3. **Function Calling 체인 (총 3회 호출)**
   - **Step 1 — get_weather 호출**
     - 입력: “서울 날씨 + 활동 추천”
     - `tool_choice`로 `get_weather`만 허용 → 모델이 자동으로 tool_call 생성
     - instructor가 툴 실행 후 결과를 모델에 재전달
   - **Step 2 — recommend_activity 호출**
     - `tool_choice`로 `recommend_activity`만 허용 → 모델이 tool_call 생성
     - instructor가 결과를 다시 모델에 전달
   - **Step 3 — 최종 답변 생성**
     - `tool_choice = "none"`
     - 모델이 도구 없이 자연어로 마무리 답변
     - 예: “현재 기온은 X도이며, Y 같은 활동을 추천합니다.”

In [ ]:
import json
import requests
from openai import OpenAI

client = OpenAI()

# =====================================================================
# 1. Tool 정의
# =====================================================================

# get_weather 툴
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "TODO: 함수 목적과 사용 상황을 설명하세요.",
    "parameters": {
        "type": "object",
        "properties": {
            # TODO: city 필드를 정의하고 설명을 채우세요.
        },
        "required": ["city"],
    },
}

# recommend_activity 툴
recommend_activity_tool = {
    "type": "function",
    "name": "recommend_activity",
    "description": "TODO: 기온 기반 추천 함수 설명을 작성하세요.",
    "parameters": {
        "type": "object",
        "properties": {
            # TODO: temperature 필드를 정의하고 설명을 채우세요.
        },
        "required": ["temperature"],
    },
}

#dummy tool
web_search_tool = {
    "type": "web_search"
    }

all_tools = [weather_tool, recommend_activity_tool, web_search_tool]


# =====================================================================
# 2. Tool 함수 구현
# =====================================================================

CITY_COORDS = {
    "서울":  (37.5665, 126.9780),
    "부산":  (35.1796, 129.0756),
    "뉴욕":  (40.7128, -74.0060),
    "도쿄":  (35.6895, 139.6917),
    "파리":  (48.8566, 2.3522),
    "두바이": (25.2048, 55.2708),
}

def get_weather(city: str):
    if city not in CITY_COORDS:
        return {"error": f"'{city}'는 현재 지원하지 않는 도시입니다."}

    lat, lon = CITY_COORDS[city]

    url = "https://api.open-meteo.com/v1/forecast"
    data = requests.get(url, params={
        "latitude": lat,
        "longitude": lon,
        "current_weather": True,
    }).json()

    temp = data["current_weather"]["temperature"]
    return {"city": city, "temperature": temp}

def recommend_activity(temperature):
    if temperature >= 25:
        return {"activity": "시원한 음료 마시면서 공원 산책하기"}
    elif temperature >= 15:
        return {"activity": "한강에서 자전거 타기"}
    elif temperature >= 5:
        return {"activity": "따뜻한 카페에서 책 읽기"}
    else:
        return {"activity": "집에서 넷플릭스 보기"}


# =====================================================================
# 3. 사용자 입력
# =====================================================================

input_list = [
    {"role": "user", "content": "지금 서울 날씨 알려줘. 그리고 그 날씨에서 하면 좋을 활동도 추천해줘."}
]


# =====================================================================
# 4. First Call — get_weather 호출
# =====================================================================

# TODO: tool_choice를 이용해 get_weather만 허용하는 첫 번째 모델 호출 코드를 작성하세요.
# - client.responses.create(...)  (model="gpt-6-luna")
# - response1/output 처리
# - tool_call 결과를 get_weather 함수에 넘기고, function_call_output을 input_list에 추가하세요.


# =====================================================================
# 5. Second Call — recommend_activity 호출
# =====================================================================

# TODO: tool_choice를 recommend_activity만 허용하도록 설정한 두 번째 모델 호출을 구현하세요.
# - client.responses.create(...)
# - response2.output을 input_list에 추가
# - tool_call 결과에서 temperature 값을 읽어 recommend_activity 함수를 실행
# - function_call_output을 input_list에 다시 추가

# =====================================================================
# 6. Third Call — 최종 자연어 응답 생성 (더 이상 도구 사용 금지)
# =====================================================================

# TODO: tool_choice="none"으로 설정하여 마지막 모델 호출을 작성하세요.
# - instructions: "지금까지 도구에서 얻은 정보만 사용..."
# - final.output_text를 출력하도록 코드를 완성하세요.

print("🌤 최종 답변:")
print(final.output_text)

### 참고 자료
- https://platform.openai.com/docs/guides/function-calling